# Module 3: Documentation Generation
**IIIT Hyderabad PG Capstone — AI-Powered Software Engineering Assistant**

**Task:** Python source code → expert natural-language documentation  
**Model:** `Salesforce/codegen-350M-multi`

### Output layout
1. Overview & System Description  
2. Architecture & Visual Breakdown  
3. Complexity Analysis Table (Markdown + LaTeX)  
4. Optimized Implementation Guide  
5. Step-by-Step Execution Trace

---
**How to run**
1. Open in Google Colab  
2. **Runtime → Change runtime type → T4 GPU**  
3. Run all cells top-to-bottom (`Runtime → Run all`)

Share this `.ipynb` file, or upload it to Google Drive and open with Colab.

## 1 — Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '⚠️ No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2 — Install packages

In [ ]:
%%capture
!pip install -q transformers==4.51.3 accelerate==1.7.0 datasets==3.6.0 \
    evaluate==0.4.3 sacrebleu==2.5.1 bert-score==0.3.13 gradio==5.31.0
print('Packages installed.')

## 3 — Setup

In [ ]:
import os, json, re, ast, urllib.request
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path('/content/doc_gen')
(ROOT / 'data').mkdir(parents=True, exist_ok=True)
(ROOT / 'results').mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'Salesforce/codegen-350M-multi'
CACHE_DIR = str(ROOT / 'model_cache')

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4 — Download CoDocBench dataset (from MBPP)

Builds code ↔ docstring pairs: **code** = MBPP solution, **docstring** = natural-language task prompt.

In [ ]:
mbpp_path = ROOT / 'data' / 'sanitized-mbpp.json'
codoc_path = ROOT / 'data' / 'codocbench_synthetic.json'

if not mbpp_path.exists():
    url = 'https://huggingface.co/datasets/Muennighoff/mbpp/resolve/main/data/sanitized-mbpp.json'
    print('Downloading MBPP…')
    urllib.request.urlretrieve(url, mbpp_path)

with open(mbpp_path) as f:
    mbpp = json.load(f)

pairs = []
for p in mbpp:
    code = p.get('code') or p.get('completion') or ''
    prompt = p.get('prompt') or p.get('text') or ''
    if not code.strip() or not prompt.strip():
        continue
    pairs.append({
        'task_id': p.get('task_id'),
        'code': code,
        'docstring': prompt,
        'source': 'mbpp_synthetic',
    })

with open(codoc_path, 'w') as f:
    json.dump(pairs, f, indent=2)

print(f'CoDocBench pairs: {len(pairs)}')
print('\n--- Sample 1 ---')
print('CODE:\n', pairs[0]['code'][:300])
print('\nREFERENCE DOC:', pairs[0]['docstring'])

## 5 — Documentation engine

AST analysis + algorithm fingerprints + CodeGen overview → **5-section expert docs**.

In [ ]:
# ── Algorithm knowledge base (common cases) ─────────────────
ALGORITHM_KB = {
    'bubble_sort': {
        'aliases': ['bubblesort', 'bubble'],
        'title': 'Bubble Sort',
        'why_named': "Larger values 'bubble' toward the end via adjacent swaps — like air bubbles rising in water.",
        'mechanics': (
            'Two nested loops. Outer pass shrinks the unsorted prefix; inner loop swaps '
            'adjacent out-of-order pairs. After pass i, the last i elements are final.'
        ),
        'best': r'$O(n)$', 'avg': r'$O(n^2)$', 'worst': r'$O(n^2)$',
        'space': r'$O(1)$', 'stable': 'Yes', 'memory': 'In-place',
        'optimized': '''def bubble_sort(arr):
    """In-place bubble sort with early-exit when a pass makes no swaps."""
    n = len(arr)
    for i in range(n - 1):
        swapped = False
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:
            break
    return arr
''',
        'kind': 'sort',
    },
    'selection_sort': {
        'aliases': ['selectionsort', 'selection'],
        'title': 'Selection Sort',
        'why_named': 'Each pass selects the minimum from the unsorted region and places it next.',
        'mechanics': 'Outer index i; inner scan finds min in [i..n); one swap into position i.',
        'best': r'$O(n^2)$', 'avg': r'$O(n^2)$', 'worst': r'$O(n^2)$',
        'space': r'$O(1)$', 'stable': 'No', 'memory': 'In-place',
        'optimized': '''def selection_sort(arr):
    n = len(arr)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if arr[j] < arr[min_idx]:
                min_idx = j
        if min_idx != i:
            arr[i], arr[min_idx] = arr[min_idx], arr[i]
    return arr
''',
        'kind': 'sort',
    },
    'insertion_sort': {
        'aliases': ['insertionsort', 'insertion'],
        'title': 'Insertion Sort',
        'why_named': 'Inserts each new element into its place in the sorted prefix — like sorting cards.',
        'mechanics': 'For each i, shift larger neighbors right until key fits.',
        'best': r'$O(n)$', 'avg': r'$O(n^2)$', 'worst': r'$O(n^2)$',
        'space': r'$O(1)$', 'stable': 'Yes', 'memory': 'In-place',
        'optimized': '''def insertion_sort(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr
''',
        'kind': 'sort',
    },
    'binary_search': {
        'aliases': ['binsearch', 'bsearch'],
        'title': 'Binary Search',
        'why_named': 'Repeatedly halves the search interval on a sorted collection.',
        'mechanics': 'Maintain [lo, hi); compare mid; discard impossible half.',
        'best': r'$O(1)$', 'avg': r'$O(\\log n)$', 'worst': r'$O(\\log n)$',
        'space': r'$O(1)$', 'stable': 'N/A', 'memory': 'In-place indices',
        'optimized': '''def binary_search(arr, target):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < target: lo = mid + 1
        elif arr[mid] > target: hi = mid
        else: return mid
    return -1
''',
        'kind': 'search',
    },
}


def _names_from_target(target):
    names = []
    if isinstance(target, ast.Name):
        names.append(target.id)
    elif isinstance(target, (ast.Tuple, ast.List)):
        for elt in target.elts:
            names.extend(_names_from_target(elt))
    elif isinstance(target, ast.Starred):
        names.extend(_names_from_target(target.value))
    return names


def _loop_depth(source_code: str) -> int:
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return 0

    def depth(node):
        if isinstance(node, (ast.For, ast.While)):
            kids = [depth(c) for c in ast.iter_child_nodes(node)]
            return 1 + (max(kids) if kids else 0)
        kids = [depth(c) for c in ast.iter_child_nodes(node)]
        return max(kids) if kids else 0

    return depth(tree)


def analyze_function(source_code: str) -> Dict[str, Any]:
    info = {
        'name': None, 'parameters': [], 'locals': [], 'returns': [],
        'has_return': False, 'has_recursion': False,
        'loop_depth': 0, 'n_loops': 0, 'n_branches': 0,
    }
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return info
    func = next((n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))), None)
    if not func:
        return info
    info['name'] = func.name
    for a in func.args.args:
        info['parameters'].append({'name': a.arg, 'kind': 'parameter'})
    param_names = {p['name'] for p in info['parameters']}
    locals_found, returns = [], []
    n_loops = n_branches = 0
    for node in ast.walk(func):
        if isinstance(node, (ast.For, ast.While)):
            n_loops += 1
        if isinstance(node, (ast.If, ast.IfExp)):
            n_branches += 1
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == func.name:
            info['has_recursion'] = True
        if isinstance(node, ast.Assign):
            for t in node.targets:
                for name in _names_from_target(t):
                    if name not in param_names and name not in locals_found:
                        locals_found.append(name)
        elif isinstance(node, ast.For):
            for name in _names_from_target(node.target):
                if name not in param_names and name not in locals_found:
                    locals_found.append(name)
        elif isinstance(node, ast.Return):
            info['has_return'] = True
            if node.value is not None:
                try:
                    returns.append(ast.unparse(node.value))
                except Exception:
                    returns.append('<expr>')
    info['locals'] = [{'name': n, 'kind': 'local variable'} for n in locals_found]
    info['returns'] = returns
    info['loop_depth'] = _loop_depth(source_code)
    info['n_loops'] = n_loops
    info['n_branches'] = n_branches
    return info


def identify_algorithm(source_code: str, func_name: Optional[str]):
    name = (func_name or '').lower().replace('-', '_')
    compact = re.sub(r'[^a-z0-9]', '', name)
    code_l = source_code.lower()
    for key, meta in ALGORITHM_KB.items():
        if key in name or key.replace('_', '') in compact:
            return key
        for alias in meta['aliases']:
            if alias in compact or alias in name:
                return key
    if _loop_depth(source_code) >= 2 and ('n-i-1' in code_l.replace(' ', '') or 'n - i - 1' in code_l):
        return 'bubble_sort'
    return None


def _complexity_table(meta):
    return (
        '| Metric | Value |\n|---|---|\n'
        f"| **Best-case time** | {meta['best']} |\n"
        f"| **Average-case time** | {meta['avg']} |\n"
        f"| **Worst-case time** | {meta['worst']} |\n"
        f"| **Space** | {meta['space']} |\n"
        f"| **Stable** | {meta['stable']} |\n"
        f"| **Memory profile** | {meta['memory']} |"
    )


def _complexity_heuristic(analysis):
    d = analysis.get('loop_depth', 0)
    if d >= 2:
        return {'best': r'$O(n^2)$', 'avg': r'$O(n^2)$', 'worst': r'$O(n^2)$',
                'space': r'$O(1)$', 'stable': 'Depends', 'memory': 'Typically in-place'}
    if d == 1:
        return {'best': r'$O(n)$', 'avg': r'$O(n)$', 'worst': r'$O(n)$',
                'space': r'$O(1)$', 'stable': 'Depends', 'memory': 'Typically in-place'}
    return {'best': r'$O(1)$', 'avg': r'$O(1)$–$O(n)$', 'worst': r'$O(n)$',
            'space': r'$O(1)$', 'stable': 'N/A', 'memory': 'Minimal'}


def _trace_bubble(sample):
    arr = sample[:]
    lines = [f'**Input:** `{arr}`', '']
    n, step = len(arr), 1
    for i in range(n - 1):
        swapped = False
        lines.append(f'**Pass {i+1}**')
        for j in range(0, n - i - 1):
            a, b = arr[j], arr[j+1]
            if a > b:
                arr[j], arr[j+1] = b, a
                swapped = True
                lines.append(f'{step}. Compare `{a}` vs `{b}` → **swap** → `{arr}`')
            else:
                lines.append(f'{step}. Compare `{a}` vs `{b}` → ordered → `{arr}`')
            step += 1
        if not swapped:
            lines.append('_No swaps — early exit._')
            break
        lines.append('')
    lines += ['', f'**Final:** `{arr}`']
    return '\n'.join(lines)


def _trace_generic(source_code, analysis, sample):
    lines = [f'**Sample dataset:** `{sample}`', '', 'Control-flow walkthrough:', '']
    try:
        tree = ast.parse(source_code)
        func = next((n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))), None)
        if func:
            for i, stmt in enumerate(func.body, 1):
                try:
                    snip = ast.unparse(stmt)
                except Exception:
                    snip = type(stmt).__name__
                if len(snip) > 100:
                    snip = snip[:97] + '...'
                lines.append(f'{i}. `{snip}`')
    except SyntaxError:
        lines.append('1. Execute source on the sample manually.')
    if analysis.get('returns'):
        lines.append(f"\n**Returns:** `{', '.join(analysis['returns'][:2])}`")
    return '\n'.join(lines)


class DocGenPipeline:
    """CodeGen wrapper for overview narrative."""

    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def generate(self, prompt, max_new_tokens=128, temperature=0.4):
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.device)
        input_len = inputs['input_ids'].shape[1]
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0,
                temperature=max(temperature, 1e-5),
                top_p=0.95,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        text = self.tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
        if '"""' in text:
            text = text[:text.find('"""')]
        return text.strip()


def generate_documentation(source_code: str, pipeline: Optional[DocGenPipeline] = None) -> str:
    """Build 5-section expert Markdown documentation."""
    analysis = analyze_function(source_code)
    algo_key = identify_algorithm(source_code, analysis.get('name'))
    algo = ALGORITHM_KB.get(algo_key) if algo_key else None

    model_text = ''
    if pipeline is not None:
        prompt = (
            '# Write a concise expert overview of this Python function.\n'
            '# Say what it does and why the name fits. 2-4 sentences. No greetings.\n\n'
            f'{source_code.strip()}\n\n# Overview:\n'
        )
        try:
            raw = pipeline.generate(prompt, max_new_tokens=120, temperature=0.4)
            model_text = ' '.join(
                ln.strip() for ln in raw.splitlines()
                if ln.strip() and not ln.strip().startswith(('def ', 'class ', '#', '>>>'))
            )
        except Exception as e:
            model_text = f'(model overview skipped: {e})'

    name = analysis.get('name') or 'module'
    if algo:
        overview = f"**{algo['title']}** (`{name}`) — {algo['why_named']}"
        if model_text:
            overview += f'\n\n{model_text}'
        architecture = algo['mechanics']
        complexity = {k: algo[k] for k in ('best', 'avg', 'worst', 'space', 'stable', 'memory')}
        optimized = algo['optimized'].rstrip()
        title = algo['title']
    else:
        overview = f"**`{name}`** — {model_text}" if model_text else (
            f"**`{name}`** transforms its inputs according to the logic below."
        )
        params = ', '.join(p['name'] for p in analysis['parameters']) or 'none'
        architecture = (
            f'Takes `{params}`. Loop depth={analysis["loop_depth"]}, '
            f'loops={analysis["n_loops"]}, branches={analysis["n_branches"]}.'
        )
        if analysis['has_recursion']:
            architecture += ' Recursive.'
        complexity = _complexity_heuristic(analysis)
        optimized = source_code.strip()
        title = name

    vars_md = '\n'.join(
        [f"- `{p['name']}` — {p['kind']}" for p in analysis['parameters']]
        + [f"- `{v['name']}` — {v['kind']}" for v in analysis['locals']]
    ) or '- (none detected)'

    sample = [5, 1, 4, 2]
    if algo_key == 'bubble_sort':
        trace = _trace_bubble(sample)
    else:
        trace = _trace_generic(source_code, analysis, sample)

    return f'''## {title}

### 1. Overview & System Description

{overview}

---

### 2. Architecture & Visual Breakdown

{architecture}

**Variables**
{vars_md}

---

### 3. Complexity Analysis

{_complexity_table(complexity)}

---

### 4. Optimized Implementation Guide

```python
{optimized}
```

---

### 5. Step-by-Step Execution Trace

{trace}
'''.strip()


print('Documentation engine ready.')

## 6 — Load CodeGen-350M

In [ ]:
print(f'Loading {MODEL_ID} on {DEVICE}…')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    low_cpu_mem_usage=True,
)
model = model.to(DEVICE)
model.eval()

pipe = DocGenPipeline(model, tokenizer, DEVICE)
print(f'Model ready ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params).')

## 7 — Demo: document `bubble_sort`

Run this cell to see the full 5-section output.

In [ ]:
from IPython.display import Markdown, display

demo_code = '''def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr
'''

print('Generating…')
doc = generate_documentation(demo_code, pipeline=pipe)
display(Markdown(doc))

## 8 — Try your own code

Paste any Python function into `my_code` and run.

In [ ]:
my_code = '''
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res)
'''

doc = generate_documentation(my_code, pipeline=pipe)
display(Markdown(doc))

## 9 — Batch eval on CoDocBench (optional)

Generates docs for N samples and scores BLEU vs reference prompts.  
Set `N_SAMPLES = 10` for a quick run; use `50` for the full report set.

In [ ]:
N_SAMPLES = 10  # change to 50 for full run

import evaluate

with open(codoc_path) as f:
    all_pairs = json.load(f)

subset = all_pairs[:N_SAMPLES]
predictions, references = [], []

print(f'Generating docs for {len(subset)} snippets…')
for i, pair in enumerate(subset):
    print(f'  [{i+1}/{len(subset)}]', end='\r', flush=True)
    doc = generate_documentation(pair['code'], pipeline=pipe)
    predictions.append(doc)
    references.append(pair['docstring'])
print('\nDone.')

bleu = evaluate.load('sacrebleu')
bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])['score'] / 100.0
print(f'BLEU: {bleu_score:.4f}')

out_path = ROOT / 'results' / 'doc_gen.json'
with open(out_path, 'w') as f:
    json.dump({
        'task': 'documentation_generation',
        'n_samples': len(subset),
        'metrics': {'bleu': bleu_score},
        'samples': [
            {'code': subset[i]['code'][:200], 'generated': predictions[i][:1500], 'reference': references[i]}
            for i in range(min(5, len(subset)))
        ],
    }, f, indent=2)
print(f'Saved → {out_path}')

# show first result
display(Markdown('### First generated sample\n\n' + predictions[0][:3000]))

## 10 — Interactive Gradio demo (optional)

Creates a small UI + public share link you can send to others.

In [ ]:
import gradio as gr

DEFAULT = '''def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr'''

def ui_generate(code):
    if not code.strip():
        return 'Please paste Python source code.', 'Empty input'
    try:
        return generate_documentation(code, pipeline=pipe), 'OK'
    except Exception as e:
        return '', f'Error: {e}'

with gr.Blocks(title='Documentation Generation') as demo:
    gr.Markdown('# Documentation Generation\nPython source → expert technical docs (CodeGen-350M)')
    with gr.Row():
        inp = gr.Code(label='Source Code', language='python', value=DEFAULT, lines=14)
        with gr.Column():
            out = gr.Markdown(label='Generated Documentation')
            status = gr.Textbox(label='Status')
    gr.Button('Generate Documentation', variant='primary').click(ui_generate, inp, [out, status])

demo.launch(share=True)  # share=True → public URL

## Share checklist

1. Download this notebook: **File → Download → Download .ipynb**  
2. Send the `.ipynb` to teammates, **or** upload to Google Drive → open with Colab  
3. Tell them: **Runtime → T4 GPU → Run all**  
4. Optional: after cell 10, copy the Gradio `*.gradio.live` link

**Project:** IIIT Hyderabad PG Capstone — Module 3 Documentation Generation